In [5]:
import numpy as np
from rdkit import Chem
from pathlib import Path
import pandas as pd

In [2]:
atomic_features: list[str] = [
    "atomic_fukui_minus",
    "atomic_fukui_plus",
    "atomic_dipole_norm",
    "atomic_quadrupole_principal_invariant_2",
    "atomic_quadrupole_principal_invariant_3",
    "atomic_polarizability_mean",
    "atomic_polarizability_anisotropy",
    "atomic_sasa",
    "partial_charge",
]

In [106]:
bond_features: list[str] = [
    "bond_length",
]

mol_features: list[str] = [
    "molecular_dipole_norm",
]

In [7]:
from scipy.constants import (
    Avogadro,  # 1/mol
    Boltzmann,  # in J/K
)

def boltzmann_weights(G: np.ndarray, T: float = 300.0) -> np.ndarray:
    k_B: float = Boltzmann * Avogadro * 0.000239005736
    delta_G = G - G.min()
    factors = np.exp(-delta_G / (k_B * T))
    return factors / factors.sum()

In [8]:
from ml_enhance import QuantumFPFileLoader

loader = QuantumFPFileLoader(Path("../data/QuantumFP/QFP_output"))
filelist = loader.list_output_files()

In [9]:
from collections.abc import Generator

def stream_conformer_df(
    file: Path,
    loader: QuantumFPFileLoader,
) -> Generator[pd.DataFrame, None, None]:
    for df in loader.stream_conformer_dataframe(file):
        df["gibbs_free_energy_300K"] = df["gibbs_free_energy"].map(lambda x: x[1][1])
        yield df

In [101]:
from data_preprocess import extract_atom_features

In [104]:
l = []
# for file in filelist[:2]:
for df in stream_conformer_df(filelist[1002], loader):
    tdf = extract_bond_features(df)
    # l.append(tdf)

# combo_df = pd.concat(l, ignore_index=True)

In [105]:
tdf #[["atom", "partial_charge"]]

,original_smiles,bond_idx,bond_length
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,0,2.650521
1,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,1,2.878536
2,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,2,3.456221
8,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,3,2.519276
6,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,4,2.061631
7,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,5,2.057901
4,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,6,2.081421
5,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,7,2.071768
3,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,8,1.820461


In [ ]:
tdf.loc[0, "original_smiles"]

'[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H:6])[H:7])[H:5]'

In [36]:
mol = Chem.MolFromSmiles('[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H:6])[H:7])[H:5]', sanitize=False)

In [ ]:
for atom in mol.GetAtoms():
    idx = atom.GetIdx()
    print(idx, atom.GetAtomMapNum(), atom.GetSymbol())


0 1 O
1 2 C
2 3 C
3 4 S
4 10 H
5 8 H
6 9 H
7 6 H
8 7 H
9 5 H


In [ ]:
mol.GetNumBonds()

9

In [38]:
mapping = {}
for bond in mol.GetBonds():
    print(bond.GetIdx(), bond.GetBeginAtom().GetAtomMapNum(), bond.GetEndAtom().GetAtomMapNum())

    mapping[(bond.GetBeginAtom().GetAtomMapNum(), bond.GetEndAtom().GetAtomMapNum())] = bond.GetIdx()

mapping

0 1 2
1 2 3
2 3 4
3 4 10
4 3 8
5 3 9
6 2 6
7 2 7
8 1 5


{(1, 2): 0,
 (2, 3): 1,
 (3, 4): 2,
 (4, 10): 3,
 (3, 8): 4,
 (3, 9): 5,
 (2, 6): 6,
 (2, 7): 7,
 (1, 5): 8}